In [1]:
import pandas as pd
import numpy as np

def objective_function(load_file='data/Load.xlsx', combined_file='data/combined.xlsx'):
    """
    Microgrid optimization objective function following the flowchart logic.
    Minimizes total cost/emissions over the time horizon.
    """
    
    # Load data
    load_data = pd.read_excel(load_file)
    combined_data = pd.read_excel(combined_file)
    
    # Initialize parameters
    T = len(load_data)
    E_pv = load_data['Solar Power'].values
    E_wt = load_data['Wind Power (50)'].values
    E_load = load_data['Community Load'].values
    E_ro = load_data['RO Load (kWh)'].values
    
    # System parameters (adjust as needed)
    E_ed = 0  # Initial stored energy
    Enet = E_pv + E_wt  # Net renewable energy
    
    # Storage parameters
    HS_capacity = 100  # Hydrogen storage capacity
    FC_capacity = 50   # Fuel cell capacity
    EL_capacity = 50   # Electrolyzer capacity
    DG_capacity = 100  # Diesel generator capacity
    
    # Cost parameters
    cost_dg = 0.5       # Diesel cost per kWh
    cost_el = 0.1       # Electrolyzer operation cost
    cost_fc = 0.15      # Fuel cell operation cost
    emission_dg = 0.8   # Emissions per kWh from DG
    
    total_cost = 0
    total_emission = 0
    
    # Time loop
    for t in range(T):
        
        # Check if renewables meet load
        if Enet[t] >= E_load[t]:
            excess = Enet[t] - E_load[t]
            E_ed += excess
            
            # Check hydrogen storage availability
            if E_ed > 0:
                # Serve by hydrogen FC
                cost_served = E_load[t] * cost_fc
                total_cost += cost_served
            else:
                # Check if excess can run electrolyzer
                if excess >= EL_capacity:
                    total_cost += EL_capacity * cost_el
                    E_ed += EL_capacity
        else:
            deficit = E_load[t] - Enet[t]
            
            # Check hydrogen storage
            if E_ed >= deficit:
                E_ed -= deficit
                total_cost += deficit * cost_fc
            else:
                # Check if DG can run
                if deficit <= DG_capacity:
                    total_cost += deficit * cost_dg
                    total_emission += deficit * emission_dg
                else:
                    # Unmet load scenario
                    total_cost += DG_capacity * cost_dg
                    total_emission += DG_capacity * emission_dg
        
        # Update energy storage
        E_ed = min(E_ed, HS_capacity)
    
    # NSGA optimization check
    if T >= 8760:
        lpsp = calculate_lpsp(E_load, Enet, E_ed)
        lpsp_max = 0.05
        
        if lpsp <= lpsp_max:
            print("System Optimized")
        else:
            print("Resize System")
    
    return total_cost, total_emission


def calculate_lpsp(load, renewable, storage):
    """Calculate Loss of Power Supply Probability"""
    total_load = np.sum(load)
    unmet_load = 0
    
    for t in range(len(load)):
        available = renewable[t] + storage
        if available < load[t]:
            unmet_load += load[t] - available
    
    lpsp = unmet_load / total_load if total_load > 0 else 0
    return lpsp




In [2]:

cost, emission = objective_function()
print(f"Total Cost: {cost:.2f}")
print(f"Total Emission: {emission:.2f}")

Resize System
Total Cost: 418765.56
Total Emission: 670024.90
